# Serving Qwen3-0.6B with vLLM: BF16 vs GPTQ-W8A16

This notebook serves two versions of the same model with **vLLM** on a local GPU and tests them side by side:

| Model | Folder | Weights |
|:--|:--|:--|
| **Qwen3-0.6B** | `Models/Qwen3-0.6B` | Original BF16 (16-bit) |
| **Qwen3-0.6B GPTQ-W8A16** | `Models/Qwen3-0.6B-GPTQ-W8A16` | 8-bit integer weights, 16-bit activations |

For each model, the notebook:
1. Starts a vLLM server and waits until it is ready
2. Sends a chat request through the OpenAI-compatible API
3. Inspects token probabilities (logprobs)
4. Reads live server metrics from `/metrics`
5. Sends concurrent requests to see **continuous batching**
6. Reuses a shared system prompt to see **prefix caching**
7. Measures generation speed
8. Streams answers with Qwen3's thinking mode off and on

At the end, the results for both models are compared.

**Why vLLM?**

| Feature | What it does |
|:--|:--|
| **Continuous batching** | Adds and removes requests at every generation step, so the GPU never waits for the slowest request |
| **PagedAttention** | Stores the KV cache in small fixed-size blocks, so almost no memory is wasted |
| **Prefix caching** | Reuses the KV cache for prompt text that several requests share |
| **Quantization support** | Serves GPTQ, AWQ, FP8 and compressed-tensors models directly |
| **OpenAI-compatible API** | Works with the standard `openai` Python client |

## Before you start

- Run this notebook with the **Python (vllm-gpu)** kernel, the WSL environment where vLLM is installed.
- Ubuntu needs a C compiler, because vLLM uses Triton to compile GPU kernels when it starts. Install it once with `sudo apt install -y build-essential`.
- Stop any other vLLM server that is already running, because this notebook starts its own server on port 8000.

> **Note: serving a model directly from Hugging Face**
>
> This notebook serves models from the local `Models/` folder. vLLM can also download and serve a model straight from the [Hugging Face Hub](https://huggingface.co/Qwen/Qwen3-0.6B) by its repository ID:
>
> ```bash
> vllm serve Qwen/Qwen3-0.6B --max-model-len 4096 --gpu-memory-utilization 0.75
> ```
>
> - On the first run, vLLM downloads the weights, tokenizer and config into `~/.cache/huggingface/hub`. Later runs reuse that cache.
> - Quantized models on the Hub are served the same way, using their repository ID instead of a local path.
> - Gated or private models need a token first: `export HF_TOKEN=hf_...`
> - To do this in this notebook, put a repository ID such as `"Qwen/Qwen3-0.6B"` in the `MODELS` dictionary below instead of a folder path.

## Step 1: Imports and settings

This cell sets the paths to the two model folders and the server options:

- **`GPU_MEMORY_UTILIZATION`**: the share of total GPU memory that vLLM reserves for weights plus KV cache. Windows also uses some GPU memory, so `0.75` leaves room.
- **`MAX_MODEL_LEN`**: the maximum number of tokens in prompt plus answer. A smaller value needs less KV cache memory.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import concurrent.futures, json, math, os, re, signal, subprocess, sys, time
from pathlib import Path

import requests
from openai import OpenAI

MODELS_DIR = Path.cwd() / "Models"
MODELS = {
    "bf16": MODELS_DIR / "Qwen3-0.6B",
    "gptq_w8a16": MODELS_DIR / "Qwen3-0.6B-GPTQ-W8A16",
}

PORT = 8000
VLLM_URL = f"http://localhost:{PORT}"
GPU_MEMORY_UTILIZATION = 0.75
MAX_MODEL_LEN = 4096

LOG_DIR = Path("logs");     LOG_DIR.mkdir(exist_ok=True)
OUT_DIR = Path("outputs");  OUT_DIR.mkdir(exist_ok=True)

results = {}   # test results for each model, filled in as the notebook runs

for name, path in MODELS.items():
    status = "found" if Path(path).exists() else "NOT FOUND (will be treated as a Hugging Face ID)"
    print(f"{name:<12} {path}  ->  {status}")

bf16         /mnt/d/Project/vLLM/vLLM/Models/Qwen3-0.6B  ->  found
gptq_w8a16   /mnt/d/Project/vLLM/vLLM/Models/Qwen3-0.6B-GPTQ-W8A16  ->  found


## Step 2: Helpers to start and stop the vLLM server

`start_server()` runs the same command you would type in a terminal:

```bash
vllm serve <model> --served-model-name <name> --port 8000 --max-model-len 4096 --gpu-memory-utilization 0.75
```

- **`<model>`**: a local folder path or a Hugging Face repository ID.
- **`--served-model-name`**: the short name that clients put in the `model` field of each request.
- **`--tokenizer`** (optional): loads the tokenizer from a different folder than the weights.
- The server output goes to a log file in `logs/`, so the notebook stays clean.
- The helper waits until `/v1/models` answers, which usually takes 30 to 90 seconds while the model loads and CUDA graphs are captured.

`read_server_stats()` takes two numbers from that log: how much GPU memory the weights use, and how many tokens fit in the KV cache.

In [2]:
VLLM_BIN = Path(sys.executable).parent / "vllm"   # the vllm command from this kernel's environment
server = None                                     # the running server process

def server_is_up():
    try:
        return requests.get(f"{VLLM_URL}/v1/models", timeout=2).status_code == 200
    except requests.ConnectionError:
        return False

def start_server(model, served_name, tokenizer=None, timeout=600):
    """Start `vllm serve` in the background and wait until it accepts requests.
    `tokenizer` optionally loads the tokenizer from a different folder or Hugging Face ID."""
    global server
    if server_is_up():
        raise RuntimeError(f"Something is already serving on port {PORT}. "
                           "Stop it first (stop_server() or `pkill -f 'vllm serve'`).")

    log_path = LOG_DIR / f"vllm_{served_name}.log"
    cmd = [str(VLLM_BIN), "serve", str(model),
           "--served-model-name", served_name,
           "--port", str(PORT),
           "--max-model-len", str(MAX_MODEL_LEN),
           "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION)]
    if tokenizer:
        cmd += ["--tokenizer", str(tokenizer)]
    print("$ vllm", " ".join(cmd[1:]).replace(str(Path.cwd()) + "/", ""))

    log_file = open(log_path, "w")
    server = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT,
                              start_new_session=True)   # own process group, so stop_server() stops all workers
    server.log_path = log_path

    start = time.time()
    while time.time() - start < timeout:
        if server.poll() is not None:                    # the process exited: show why
            print(log_path.read_text()[-3000:])
            raise RuntimeError(f"vLLM server exited. Full log: {log_path}")
        if server_is_up():
            print(f"Server ready after {time.time() - start:.0f}s  (log: {log_path})")
            return
        time.sleep(3)
    raise TimeoutError(f"Server not ready after {timeout}s. Check {log_path}")

def stop_server():
    """Stop the server and free its GPU memory."""
    global server
    if server is None or server.poll() is not None:
        print("No server running.")
        return
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
    try:
        server.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(os.getpgid(server.pid), signal.SIGKILL)
        server.wait()
    while server_is_up():
        time.sleep(1)
    print("Server stopped.")
    server = None

def read_server_stats():
    """Read weight memory and KV cache capacity from the server log."""
    text = server.log_path.read_text()
    mem = re.search(r"Model loading took ([\d.]+) GiB", text)
    kv = re.search(r"GPU KV cache size: ([\d,]+) tokens", text)
    return {"weights_gib": float(mem.group(1)) if mem else None,
            "kv_cache_tokens": int(kv.group(1).replace(",", "")) if kv else None}

## Step 3: Test helpers

Each test is a small function, so you can run exactly the same tests on both models.

All requests use the standard `openai` client pointed at the local server. vLLM accepts any API key, so `"unused"` works.

Qwen3 has a **thinking mode** that writes its reasoning inside `<think>...</think>` before answering. Most tests turn it off with `chat_template_kwargs: {"enable_thinking": False}` to keep answers short and timings comparable.

In [3]:
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

def chat(prompt, model, system=None, **kwargs):
    """Send one chat request and return the response."""
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]
    extra = {**NO_THINKING, **kwargs.pop("extra_body", {})}
    return client.chat.completions.create(model=model, messages=messages,
                                          extra_body=extra, **kwargs)

def get_vllm_metrics():
    """Read the Prometheus /metrics page and return {metric_name: value}."""
    metrics = {}
    for line in requests.get(f"{VLLM_URL}/metrics").text.splitlines():
        if line.startswith("#") or not line.strip():
            continue
        name = line.split("{")[0].split()[0]
        try:
            metrics[name] = float(line.split()[-1])
        except ValueError:
            pass
    return metrics

**Test A: first request.** Sends one question and measures the round-trip time and token counts.

- `temperature=0.7`, `top_p=0.8` and `top_k=20` are the sampling settings Qwen recommends for non-thinking mode.
- `usage` shows how many tokens were in the prompt and in the answer.

In [4]:
def test_first_request(model):
    start = time.time()
    resp = chat("What is PagedAttention in one sentence?", model,
                max_tokens=80, temperature=0.7, top_p=0.8, extra_body={"top_k": 20})
    elapsed = time.time() - start

    print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
    print(resp.choices[0].message.content)
    print(f"\nUsage: {resp.usage.prompt_tokens} prompt + {resp.usage.completion_tokens} "
          f"completion = {resp.usage.total_tokens} total")
    return {"latency_s": round(elapsed, 3), "answer": resp.choices[0].message.content}

**Test B: token probabilities.** With `logprobs=True`, vLLM returns the log-probability of each generated token, and with `top_logprobs=5` it also returns the 5 most likely alternatives.

- `temperature=0.0` always picks the most likely token, so the answer is repeatable.
- The bars show how confident the model is. Comparing these numbers between the two models shows how much 8-bit quantization changes the model's predictions.

In [5]:
def test_logprobs(model, prompt="The capital of France is"):
    resp = chat(prompt, model, max_tokens=15, temperature=0.0,
                logprobs=True, top_logprobs=5)
    print(f"Response: {resp.choices[0].message.content.strip()}\n")
    print("Token-by-token probabilities:\n")

    tokens = resp.choices[0].logprobs.content
    for tok in tokens[:8]:
        print(f"  Chosen: {tok.token!r}  (logprob {tok.logprob:.2f})")
        for alt in tok.top_logprobs[:5]:
            pct = math.exp(alt.logprob) * 100
            bar = "█" * min(20, max(1, int(pct / 5)))
            print(f"    {pct:5.1f}%  {bar}  {alt.token!r}")
        print()
    return {"answer": resp.choices[0].message.content.strip(),
            "mean_logprob": round(sum(t.logprob for t in tokens) / len(tokens), 4)}

**Test C: server metrics.** vLLM publishes live counters at `/metrics` in Prometheus format. The most useful ones are:

- **`num_requests_running` / `num_requests_waiting`**: requests being processed vs. queued
- **`kv_cache_usage_perc`**: how full the KV cache is, from 0 to 1
- **`prompt_tokens_total` / `generation_tokens_total`**: total tokens processed so far

In [6]:
METRIC_KEYS = ["vllm:num_requests_running", "vllm:num_requests_waiting",
               "vllm:kv_cache_usage_perc", "vllm:gpu_cache_usage_perc",
               "vllm:prompt_tokens_total", "vllm:generation_tokens_total"]

def test_metrics(label):
    metrics = get_vllm_metrics()
    print("Current vLLM metrics:")
    for key in METRIC_KEYS:
        if key in metrics:
            print(f"  {key.replace('vllm:', ''):<26} {metrics[key]:g}")
    path = OUT_DIR / f"metrics_{label}.json"
    path.write_text(json.dumps(metrics, indent=2))
    print(f"\nAll metrics saved to {path}")

**Test D: continuous batching.** Sends 5 requests at the same time. vLLM runs them together in one batch: each generation step produces one token for every active request, and a finished request frees its slot right away.

The helper checks `/metrics` repeatedly while the requests run and records the **peak** number running at once.

In [7]:
BATCH_PROMPTS = [
    "What is quantization?",
    "Explain KV caching briefly.",
    "What is continuous batching?",
    "Why is LLM inference memory-bound?",
    "What is PagedAttention?",
]

def test_concurrent(model):
    before = get_vllm_metrics()
    print(f"Sending {len(BATCH_PROMPTS)} concurrent requests...\n")
    start = time.time()
    peak_running = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=len(BATCH_PROMPTS)) as pool:
        futures = {pool.submit(chat, p, model, max_tokens=60, temperature=0.7): p
                   for p in BATCH_PROMPTS}
        while not all(f.done() for f in futures):            # sample while requests run
            peak_running = max(peak_running,
                               get_vllm_metrics().get("vllm:num_requests_running", 0))
            time.sleep(0.05)
        for f in futures:
            print(f"  done: {futures[f][:40]!r:<44} -> {f.result().usage.completion_tokens} tokens")

    elapsed = time.time() - start
    tokens = get_vllm_metrics()["vllm:generation_tokens_total"] - before["vllm:generation_tokens_total"]
    print(f"\nPeak requests running together: {peak_running:g}")
    print(f"All {len(BATCH_PROMPTS)} finished in {elapsed:.2f}s  |  "
          f"{tokens:g} tokens  |  {tokens / elapsed:.1f} tokens/s")
    return {"elapsed_s": round(elapsed, 3), "peak_running": peak_running,
            "tokens_per_s": round(tokens / elapsed, 1)}

**Test E: prefix caching.** Sends 5 different questions that share the same system prompt.

- The first request computes the KV cache for the system prompt.
- The following requests find those blocks already cached and skip recomputing them.
- `prefix_cache_hits` counts how many prompt tokens were reused from the cache.

With a short system prompt the time saved is small. With system prompts that are thousands of tokens long, prefix caching removes most of the prompt processing work.

In [8]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that explains LLM inference and serving. "
    "You specialize in quantization, KV caching, batching and model serving. "
    "Keep every answer short: one or two sentences."
)
PREFIX_QUESTIONS = [
    "What is weight quantization?",
    "How does vLLM handle memory?",
    "What is continuous batching?",
    "Why use prefix caching?",
    "What is GPTQ?",
]

def test_prefix_caching(model):
    before = get_vllm_metrics()
    print("Sending 5 requests with the same system prompt...\n")
    for i, q in enumerate(PREFIX_QUESTIONS, 1):
        t0 = time.time()
        resp = chat(q, model, system=SYSTEM_PROMPT, max_tokens=60, temperature=0.7)
        dt, n = time.time() - t0, resp.usage.completion_tokens
        print(f"  [{i}] {q:<32} {dt:.2f}s  ({n} tok, {dt / max(n, 1) * 1000:.0f} ms/tok)")
    after = get_vllm_metrics()

    queries = after.get("vllm:prefix_cache_queries_total", 0) - before.get("vllm:prefix_cache_queries_total", 0)
    hits = after.get("vllm:prefix_cache_hits_total", 0) - before.get("vllm:prefix_cache_hits_total", 0)
    rate = hits / queries * 100 if queries else 0
    print(f"\nPrompt tokens looked up in the prefix cache: {queries:g}")
    print(f"Prompt tokens reused from the cache:         {hits:g}  ({rate:.0f}% hit rate)")
    return {"prefix_hit_rate_pct": round(rate, 1)}

**Test F: generation speed.** Generates exactly 256 tokens for one request and measures tokens per second.

- `ignore_eos=True` makes the model keep going past its normal stopping point, so every run produces the same number of tokens and the speeds are comparable.
- Generating tokens one at a time is **memory-bound**: each step reads all the weights from GPU memory. Smaller weights, such as 8-bit instead of 16-bit, mean less data to read per step.
- The test runs 3 times and reports the median, after one warm-up run.

In [9]:
def test_speed(model, n_tokens=256, runs=3):
    def one_run():
        t0 = time.time()
        resp = chat("Write a detailed explanation of how GPUs run neural networks.", model,
                    max_tokens=n_tokens, temperature=0.0,
                    extra_body={"ignore_eos": True})
        return resp.usage.completion_tokens / (time.time() - t0)

    one_run()                                   # warm-up
    speeds = sorted(one_run() for _ in range(runs))
    median = speeds[len(speeds) // 2]
    print(f"Single-request speed over {runs} runs of {n_tokens} tokens: "
          + ", ".join(f"{s:.1f}" for s in speeds) + " tokens/s")
    print(f"Median: {median:.1f} tokens/s")
    return {"single_tokens_per_s": round(median, 1)}

**Test G: thinking mode.** Streams the same question twice, once with thinking off and once with it on.

- With `stream=True`, tokens are printed as they arrive instead of all at the end.
- Thinking mode usually gives better answers to hard questions, but it uses many more tokens, which means more KV cache, more compute and a longer wait.

In [10]:
def test_thinking(model, prompt="What makes continuous batching better than static batching?"):
    for label, thinking, max_tok in [("Thinking OFF", False, 80), ("Thinking ON", True, 200)]:
        print(f"=== {label} ===\n")
        start, chunks = time.time(), 0
        stream = client.chat.completions.create(
            model=model, messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tok, temperature=0.7, stream=True,
            extra_body={"chat_template_kwargs": {"enable_thinking": thinking}})
        for chunk in stream:
            text = chunk.choices[0].delta.content if chunk.choices else None
            if text:
                sys.stdout.write(text); sys.stdout.flush()
                chunks += 1
        print(f"\n  [{chunks} tokens, {time.time() - start:.2f}s]\n")

**Quality check.** Asks a few questions that have a single correct answer, with `temperature=0` so the output is repeatable. Running the same questions on both models shows whether quantization changed any answers.

In [11]:
QUALITY_PROMPTS = [
    "What is 17 * 23? Answer with the number only.",
    "What is the chemical symbol for gold? Answer with the symbol only.",
    "Who wrote 'Pride and Prejudice'? Answer with the name only.",
    "What is the largest planet in the solar system? Answer with one word.",
]

def test_quality(model):
    answers = {}
    for p in QUALITY_PROMPTS:
        answers[p] = chat(p, model, max_tokens=20, temperature=0.0).choices[0].message.content.strip()
        print(f"  {p:<72} -> {answers[p]}")
    return {"quality_answers": answers}

## Step 4: Serve and test the BF16 model

This starts the server with the original 16-bit model. The first start takes longer because vLLM compiles the model and captures CUDA graphs. Later starts reuse the compile cache.

In [12]:
BF16 = "qwen3-0.6b-bf16"
start_server(MODELS["bf16"], BF16)

results[BF16] = {"disk_gb": round(sum(f.stat().st_size for f in Path(MODELS["bf16"]).glob("*.safetensors")) / 1e9, 2)}
results[BF16].update(read_server_stats())
print(results[BF16])

$ vllm serve Models/Qwen3-0.6B --served-model-name qwen3-0.6b-bf16 --port 8000 --max-model-len 4096 --gpu-memory-utilization 0.75


Server ready after 63s  (log: logs/vllm_qwen3-0.6b-bf16.log)
{'disk_gb': 1.5, 'weights_gib': 1.12, 'kv_cache_tokens': 26144}


The server is running. `/v1/models` lists the model under the name passed to `--served-model-name`:

In [13]:
print(json.dumps(requests.get(f"{VLLM_URL}/v1/models").json()["data"][0], indent=2)[:600])

{
  "id": "qwen3-0.6b-bf16",
  "object": "model",
  "created": 1794939876,
  "owned_by": "vllm",
  "root": "/mnt/d/Project/vLLM/vLLM/Models/Qwen3-0.6B",
  "parent": null,
  "max_model_len": 4096,
  "permission": [
    {
      "id": "modelperm-afda5b3c453067a7",
      "object": "model_permission",
      "created": 1794939876,
      "allow_create_engine": false,
      "allow_sampling": true,
      "allow_logprobs": true,
      "allow_search_indices": false,
      "allow_view": true,
      "allow_fine_tuning": false,
      "organization": "*",
      "group": null,
      "is_blocking": false
    }


### A. First request

In [14]:
results[BF16].update(test_first_request(BF16))

Response (1.00s, 32 tokens):
Paged Attention is a technique that allows for efficient processing of large-scale text by dividing the input into smaller segments and applying attention mechanisms to each segment independently.

Usage: 21 prompt + 32 completion = 53 total


### B. Token probabilities

In [15]:
results[BF16].update(test_logprobs(BF16))

Response: The capital of France is **Paris**.

Token-by-token probabilities:

  Chosen: 'The'  (logprob -0.00)
     99.9%  ███████████████████  'The'
      0.1%  █  'France'
      0.0%  █  'Capital'
      0.0%  █  'Paris'
      0.0%  █  'Le'

  Chosen: ' capital'  (logprob -0.00)
    100.0%  ███████████████████  ' capital'
      0.0%  █  ' Capital'
      0.0%  █  ' **'
      0.0%  █  ' French'
      0.0%  █  '**'

  Chosen: ' of'  (logprob -0.00)
    100.0%  ███████████████████  ' of'
      0.0%  █  ' city'
      0.0%  █  ' is'
      0.0%  █  ' ('
      0.0%  █  ' and'

  Chosen: ' France'  (logprob -0.00)
    100.0%  ███████████████████  ' France'
      0.0%  █  ' **'
      0.0%  █  'France'
      0.0%  █  ' the'
      0.0%  █  ' Europe'

  Chosen: ' is'  (logprob -0.00)
    100.0%  ███████████████████  ' is'
      0.0%  █  ' in'
      0.0%  █  ','
      0.0%  █  ' was'
      0.0%  █  ' **'

  Chosen: ' **'  (logprob -0.09)
     91.2%  ██████████████████  ' **'
      7.5%  █  ' Paris'

### C. Server metrics

In [16]:
test_metrics("bf16")

Current vLLM metrics:
  num_requests_running       0
  num_requests_waiting       0
  kv_cache_usage_perc        0
  prompt_tokens_total        38
  generation_tokens_total    42

All metrics saved to outputs/metrics_bf16.json


### D. Continuous batching

> Running the cell a second time can show a higher peak, because the first run also warms up the server.

In [17]:
results[BF16].update(test_concurrent(BF16))

Sending 5 concurrent requests...



  done: 'What is quantization?'                      -> 60 tokens
  done: 'Explain KV caching briefly.'                -> 60 tokens
  done: 'What is continuous batching?'               -> 60 tokens
  done: 'Why is LLM inference memory-bound?'         -> 60 tokens
  done: 'What is PagedAttention?'                    -> 60 tokens

Peak requests running together: 5
All 5 finished in 0.81s  |  300 tokens  |  370.1 tokens/s


### E. Prefix caching

In [18]:
results[BF16].update(test_prefix_caching(BF16))

Sending 5 requests with the same system prompt...



  [1] What is weight quantization?     0.37s  (38 tok, 10 ms/tok)
  [2] How does vLLM handle memory?     0.20s  (20 tok, 10 ms/tok)


  [3] What is continuous batching?     0.21s  (21 tok, 10 ms/tok)
  [4] Why use prefix caching?          0.16s  (15 tok, 11 ms/tok)


  [5] What is GPTQ?                    0.24s  (25 tok, 10 ms/tok)

Prompt tokens looked up in the prefix cache: 300
Prompt tokens reused from the cache:         128  (43% hit rate)


### F. Generation speed

In [19]:
results[BF16].update(test_speed(BF16))

Single-request speed over 3 runs of 256 tokens: 114.5, 114.5, 114.8 tokens/s
Median: 114.5 tokens/s


### G. Thinking mode

In [20]:
test_thinking(BF16)

=== Thinking OFF ===

Continuous

 batching

 is

 generally

 considered

 more

... (267 outputs left out)

 It

's

 not


  [200 tokens, 1.82s]



### Quality check

In [21]:
results[BF16].update(test_quality(BF16))

  What is 17 * 23? Answer with the number only.                            -> 17 × 23 = **391**.


  What is the chemical symbol for gold? Answer with the symbol only.       -> Au


  Who wrote 'Pride and Prejudice'? Answer with the name only.              -> Pride and Prejudice
  What is the largest planet in the solar system? Answer with one word.    -> Mercury


Stop the BF16 server to free GPU memory for the next model:

In [22]:
stop_server()

Server stopped.


## Step 5: Serve and test the GPTQ-W8A16 model

**GPTQ-W8A16** means:
- **W8**: the weights of every linear layer are stored as 8-bit integers instead of 16-bit floats, which halves their size.
- **A16**: activations, the values flowing between layers, stay in 16-bit.
- **GPTQ**: the method used to choose the 8-bit values. It corrects rounding errors layer by layer using sample data, so accuracy stays close to the original.

The model folder contains a `quantization_config` in `config.json`. vLLM reads it and picks a fast kernel for the format automatically, so the `vllm serve` command is exactly the same as for the BF16 model.

In [23]:
config = json.loads((Path(MODELS["gptq_w8a16"]) / "config.json").read_text())
weights = config["quantization_config"]["config_groups"]["group_0"]["weights"]
print("quant_method:", config["quantization_config"]["quant_method"])
print("weights:     ", {k: weights[k] for k in ["num_bits", "type", "strategy", "group_size", "symmetric"]})
print("ignored:     ", config["quantization_config"]["ignore"])

quant_method: compressed-tensors
weights:      {'num_bits': 8, 'type': 'int', 'strategy': 'group', 'group_size': 128, 'symmetric': True}
ignored:      ['lm_head']


`lm_head`, the final layer that picks the next token, is kept in 16-bit because it is the most sensitive to quantization.

**Why `--tokenizer` points to the BF16 folder:** the tool that saved this quantized model wrote `tokenizer_config.json` in the newer Transformers v5 format, which stores `extra_special_tokens` as a list. The Transformers 4.x version that vLLM 0.18.0 uses expects a dictionary there and fails to load it. Quantization doesn't change the tokenizer, so the original model's tokenizer is identical and loads without problems.

Start the server:

In [24]:
GPTQ = "qwen3-0.6b-gptq-w8a16"
start_server(MODELS["gptq_w8a16"], GPTQ, tokenizer=MODELS["bf16"])

results[GPTQ] = {"disk_gb": round(sum(f.stat().st_size for f in Path(MODELS["gptq_w8a16"]).glob("*.safetensors")) / 1e9, 2)}
results[GPTQ].update(read_server_stats())
print(results[GPTQ])

$ vllm serve Models/Qwen3-0.6B-GPTQ-W8A16 --served-model-name qwen3-0.6b-gptq-w8a16 --port 8000 --max-model-len 4096 --gpu-memory-utilization 0.75 --tokenizer Models/Qwen3-0.6B


Server ready after 75s  (log: logs/vllm_qwen3-0.6b-gptq-w8a16.log)
{'disk_gb': 0.76, 'weights_gib': 0.74, 'kv_cache_tokens': 29632}


Now run the same tests as before:

### A. First request

In [25]:
results[GPTQ].update(test_first_request(GPTQ))

Response (0.59s, 27 tokens):
Paged Attention is a technique that allows attention to be distributed across multiple tokens in a sequence, improving efficiency and performance in models.

Usage: 21 prompt + 27 completion = 48 total


### B. Token probabilities

In [26]:
results[GPTQ].update(test_logprobs(GPTQ))

Response: The capital of France is **Paris**.

Token-by-token probabilities:

  Chosen: 'The'  (logprob -0.00)
     99.8%  ███████████████████  'The'
      0.1%  █  'France'
      0.0%  █  'Capital'
      0.0%  █  'Paris'
      0.0%  █  'Le'

  Chosen: ' capital'  (logprob -0.00)
    100.0%  ███████████████████  ' capital'
      0.0%  █  ' Capital'
      0.0%  █  ' **'
      0.0%  █  ' French'
      0.0%  █  '**'

  Chosen: ' of'  (logprob -0.00)
    100.0%  ███████████████████  ' of'
      0.0%  █  ' city'
      0.0%  █  ' is'
      0.0%  █  ' ('
      0.0%  █  ' and'

  Chosen: ' France'  (logprob -0.00)
    100.0%  ███████████████████  ' France'
      0.0%  █  ' **'
      0.0%  █  ' the'
      0.0%  █  'France'
      0.0%  █  ' Europe'

  Chosen: ' is'  (logprob -0.00)
    100.0%  ███████████████████  ' is'
      0.0%  █  ' in'
      0.0%  █  ','
      0.0%  █  ' was'
      0.0%  █  ' **'

  Chosen: ' **'  (logprob -0.09)
     91.1%  ██████████████████  ' **'
      7.5%  █  ' Paris'

### C. Server metrics

In [27]:
test_metrics("gptq_w8a16")

Current vLLM metrics:
  num_requests_running       0
  num_requests_waiting       0
  kv_cache_usage_perc        0
  prompt_tokens_total        38
  generation_tokens_total    37

All metrics saved to outputs/metrics_gptq_w8a16.json


### D. Continuous batching

In [28]:
results[GPTQ].update(test_concurrent(GPTQ))

Sending 5 concurrent requests...



  done: 'What is quantization?'                      -> 60 tokens
  done: 'Explain KV caching briefly.'                -> 60 tokens
  done: 'What is continuous batching?'               -> 60 tokens
  done: 'Why is LLM inference memory-bound?'         -> 60 tokens
  done: 'What is PagedAttention?'                    -> 60 tokens

Peak requests running together: 5
All 5 finished in 0.61s  |  300 tokens  |  492.2 tokens/s


### E. Prefix caching

In [29]:
results[GPTQ].update(test_prefix_caching(GPTQ))

Sending 5 requests with the same system prompt...



  [1] What is weight quantization?     0.24s  (27 tok, 9 ms/tok)


  [2] How does vLLM handle memory?     0.25s  (36 tok, 7 ms/tok)
  [3] What is continuous batching?     0.16s  (21 tok, 8 ms/tok)


  [4] Why use prefix caching?          0.11s  (15 tok, 7 ms/tok)
  [5] What is GPTQ?                    0.18s  (25 tok, 7 ms/tok)

Prompt tokens looked up in the prefix cache: 300
Prompt tokens reused from the cache:         128  (43% hit rate)


### F. Generation speed

In [30]:
results[GPTQ].update(test_speed(GPTQ))

Single-request speed over 3 runs of 256 tokens: 161.0, 161.0, 161.2 tokens/s
Median: 161.0 tokens/s


### G. Thinking mode

In [31]:
test_thinking(GPTQ)

=== Thinking OFF ===

Continuous

 batching

 is

 generally

 considered

 more

... (268 outputs left out)

 Continuous

 batching

 might


  [200 tokens, 1.31s]



### Quality check

In [32]:
results[GPTQ].update(test_quality(GPTQ))

  What is 17 * 23? Answer with the number only.                            -> 17 × 23 = 391.


  What is the chemical symbol for gold? Answer with the symbol only.       -> Au


  Who wrote 'Pride and Prejudice'? Answer with the name only.              -> Pride and Prejudice


  What is the largest planet in the solar system? Answer with one word.    -> Mercury


In [33]:
stop_server()

Server stopped.


## Step 6: Compare the two models

This table puts the numbers from both runs side by side:

- **Weights on disk / in GPU memory**: 8-bit weights should take about half the space.
- **KV cache capacity**: memory the weights don't use goes to the KV cache, so the quantized model can hold more tokens at once.
- **Speed**: single-request and concurrent throughput in tokens per second.
- **Mean logprob**: the model's average confidence on the logprobs prompt. Values close together mean quantization changed the model's predictions very little.

In [34]:
ROWS = [
    ("Weights on disk (GB)",             "disk_gb"),
    ("Weights in GPU memory (GiB)",      "weights_gib"),
    ("KV cache capacity (tokens)",       "kv_cache_tokens"),
    ("First request latency (s)",        "latency_s"),
    ("Single-request speed (tok/s)",     "single_tokens_per_s"),
    ("5 concurrent requests (tok/s)",    "tokens_per_s"),
    ("Peak requests running together",   "peak_running"),
    ("Prefix cache hit rate (%)",        "prefix_hit_rate_pct"),
    ("Mean logprob ('capital of France')", "mean_logprob"),
]

def fmt(v):
    if v is None: return "n/a"
    return f"{v:,}" if isinstance(v, int) else f"{v:,.4g}" if isinstance(v, float) else str(v)

print(f"{'Metric':<38}{'BF16':>14}{'GPTQ-W8A16':>14}{'Change':>10}")
print("-" * 76)
for label, key in ROWS:
    a, b = results.get(BF16, {}).get(key), results.get(GPTQ, {}).get(key)
    change = f"{(b - a) / abs(a) * 100:+.0f}%" if isinstance(a, (int, float)) and isinstance(b, (int, float)) and a else ""
    print(f"{label:<38}{fmt(a):>14}{fmt(b):>14}{change:>10}")

Metric                                          BF16    GPTQ-W8A16    Change
----------------------------------------------------------------------------
Weights on disk (GB)                             1.5          0.76      -49%
Weights in GPU memory (GiB)                     1.12          0.74      -34%
KV cache capacity (tokens)                    26,144        29,632      +13%
First request latency (s)                          1         0.588      -41%
Single-request speed (tok/s)                   114.5           161      +41%
5 concurrent requests (tok/s)                  370.1         492.2      +33%
Peak requests running together                     5             5       +0%
Prefix cache hit rate (%)                       42.7          42.7       +0%
Mean logprob ('capital of France')           -0.0212        -0.022       -4%


Answers to the quality check questions from both models:

In [35]:
same = 0
for p in QUALITY_PROMPTS:
    a = results[BF16]["quality_answers"][p]
    b = results[GPTQ]["quality_answers"][p]
    same += a == b
    print(f"{'same' if a == b else 'DIFF':<5} {p}\n      BF16: {a}\n      GPTQ: {b}\n")
print(f"{same}/{len(QUALITY_PROMPTS)} answers identical")

DIFF  What is 17 * 23? Answer with the number only.
      BF16: 17 × 23 = **391**.
      GPTQ: 17 × 23 = 391.

same  What is the chemical symbol for gold? Answer with the symbol only.
      BF16: Au
      GPTQ: Au

same  Who wrote 'Pride and Prejudice'? Answer with the name only.
      BF16: Pride and Prejudice
      GPTQ: Pride and Prejudice

same  What is the largest planet in the solar system? Answer with one word.
      BF16: Mercury
      GPTQ: Mercury

3/4 answers identical


**Reading the results:** the quantized model should give the **same** answers as the original, not necessarily **correct** ones. A 0.6B model is small and can get simple facts wrong. When both models make the same mistake, that comes from the base model, not from quantization. Small differences in wording, such as `**391**` vs `391`, are normal: 8-bit weights shift the probabilities slightly, which can change formatting tokens.

Save all results to a JSON file, so you can compare later runs or other models:

In [36]:
path = OUT_DIR / "serving_test_results.json"
path.write_text(json.dumps(results, indent=2))
print(f"Saved to {path}")

Saved to outputs/serving_test_results.json


## Step 7 (optional): How big is the KV cache?

For every token in context, each attention layer stores a **key** and a **value** vector for every KV head. The size per token is:

```
2 (key + value) x layers x KV heads x head size x bytes per value
```

The numbers come from the model's `config.json`. GPTQ-W8A16 only quantizes the weights, so the KV cache is still 16-bit and the same size for both models. The quantized model fits more tokens only because its weights leave more memory free.

In [37]:
cfg = json.loads((Path(MODELS["bf16"]) / "config.json").read_text())
layers, kv_heads, head_dim = cfg["num_hidden_layers"], cfg["num_key_value_heads"], cfg["head_dim"]
dtype_bytes = 2   # BF16

per_token = 2 * layers * kv_heads * head_dim * dtype_bytes
print(f"KV cache per token: 2 x {layers} x {kv_heads} x {head_dim} x {dtype_bytes} "
      f"= {per_token:,} bytes ({per_token // 1024} KB)\n")
print(f"  {'Context':>8}  {'KV cache':>10}")
for ctx in [1, 64, 256, 1024, 4096]:
    size = per_token * ctx
    print(f"  {ctx:>7}t  {size / 1024**2:>8.1f} MB")
print(f"\n  10 requests x 4096 tokens = {per_token * 4096 * 10 / 1024**3:.2f} GB")

KV cache per token: 2 x 28 x 8 x 128 x 2 = 114,688 bytes (112 KB)

   Context    KV cache
        1t       0.1 MB
       64t       7.0 MB
      256t      28.0 MB
     1024t     112.0 MB
     4096t     448.0 MB

  10 requests x 4096 tokens = 4.38 GB


## Summary

- Served **Qwen3-0.6B** in BF16 and in **GPTQ-W8A16** with the same `vllm serve` command. vLLM detects the quantization format from `config.json`.
- Queried both through the **OpenAI-compatible API** with the standard `openai` client.
- Used **logprobs** and fixed-answer questions to check how close the quantized model stays to the original.
- Read live **server metrics** from `/metrics`.
- Saw **continuous batching** run several requests together and **prefix caching** reuse a shared system prompt.
- Compared weight size, KV cache capacity and generation speed.


**Resources:** [vLLM on GitHub](https://github.com/vllm-project/vllm) · [vLLM docs](https://docs.vllm.ai/en/latest/) · [Quantization in vLLM](https://docs.vllm.ai/en/latest/features/quantization/)